In [ ]:
"""
02_content_based.py — Sistema de Recomendación Basado en Contenido
===================================================================

Este módulo implementa un sistema de recomendación Content-Based que
sugiere películas similares analizando sus características textuales.

Técnicas implementadas:
1. TF-IDF sobre sinopsis (overview) + Similitud Coseno
2. Metadata Soup (géneros + keywords + overview) + CountVectorizer
3. Sistema combinado con filtros avanzados

Conceptos clave:
- TF-IDF (Term Frequency - Inverse Document Frequency)
- Similitud Coseno
- Vectorización de texto
- Content-Based Filtering

Ejecutar:
    python 02_content_based.py
"""

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from utils import load_and_clean_data, create_soup, fuzzy_match_title

In [ ]:
# =============================================================================
# Configuración
# =============================================================================
OUTPUT_DIR = "output"

In [ ]:
def ensure_output_dir():
    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)

=============================================================================
PARTE 1: TF-IDF sobre Sinopsis (Overview)
=============================================================================

In [ ]:
def explain_tfidf():
    """
    Imprime una explicación teórica de TF-IDF.
    """
    print("""
    ╔══════════════════════════════════════════════════════════════════════╗
    ║                     TF-IDF: TEORÍA                                ║
    ╠══════════════════════════════════════════════════════════════════════╣
    ║                                                                    ║
    ║  TF-IDF = Term Frequency × Inverse Document Frequency              ║
    ║                                                                    ║
    ║  ┌─────────────────────────────────────────────────────────────┐   ║
    ║  │                                                            │   ║
    ║  │  TF(t,d) = frecuencia del término t en documento d         │   ║
    ║  │            ────────────────────────────────────────         │   ║
    ║  │            total de términos en documento d                 │   ║
    ║  │                                                            │   ║
    ║  │  IDF(t) = log( N / df(t) )                                 │   ║
    ║  │           N = total de documentos                          │   ║
    ║  │           df(t) = documentos que contienen t               │   ║
    ║  │                                                            │   ║
    ║  │  TF-IDF(t,d) = TF(t,d) × IDF(t)                           │   ║
    ║  │                                                            │   ║
    ║  │  → Palabras frecuentes en UN documento pero raras          │   ║
    ║  │    globalmente reciben alto score                          │   ║
    ║  │  → Palabras comunes en TODOS los documentos (the, a, is)   │   ║
    ║  │    reciben bajo score                                      │   ║
    ║  └─────────────────────────────────────────────────────────────┘   ║
    ║                                                                    ║
    ║  Ejemplo:                                                         ║
    ║    "robot" en una sinopsis de ciencia ficción → alto TF-IDF       ║
    ║    "the" en cualquier sinopsis → bajo TF-IDF (muy común)          ║
    ╚══════════════════════════════════════════════════════════════════════╝
    """)

In [ ]:
def build_tfidf_model(df):
    """
    Construye el modelo TF-IDF sobre las sinopsis de películas.

    Pasos:
    1. Crear TfidfVectorizer con parámetros optimizados
    2. Ajustar y transformar los overviews → matriz TF-IDF
    3. Calcular matriz de similitud coseno

    Parámetros:
        df (pd.DataFrame): DataFrame con columna 'overview'

    Retorna:
        tuple: (tfidf_matrix, cosine_sim, tfidf_vectorizer)
    """
    print("\n🔧 Construyendo modelo TF-IDF sobre sinopsis...")

    # Configurar el vectorizador
    tfidf = TfidfVectorizer(
        max_features=20000,         # Limitar vocabulario para eficiencia
        stop_words="english",       # Remover stopwords en inglés
        ngram_range=(1, 2),         # Unigramas y bigramas
        min_df=2,                   # Ignorar términos que aparecen en <2 docs
        max_df=0.95,                # Ignorar términos que aparecen en >95% docs
        sublinear_tf=True,          # Aplicar log(1+tf) en vez de tf crudo
    )

    # Transformar overviews a matriz TF-IDF
    tfidf_matrix = tfidf.fit_transform(df["overview"])

    print(f"   📐 Forma de la matriz TF-IDF: {tfidf_matrix.shape}")
    print(f"   📝 Vocabulario: {len(tfidf.vocabulary_):,} términos únicos")
    print(f"   💾 Densidad de la matriz: {tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]) * 100:.4f}%")

    # Calcular similitud coseno
    print("   ⏳ Calculando similitud coseno (esto puede tomar unos segundos)...")
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    print(f"   ✅ Matriz de similitud: {cosine_sim.shape}")

    return tfidf_matrix, cosine_sim, tfidf

In [ ]:
def get_recommendations_tfidf(title, df, cosine_sim, n=10):
    """
    Obtiene las N películas más similares a un título dado,
    usando la matriz de similitud coseno basada en TF-IDF.

    Similitud Coseno:
        cos(A, B) = (A · B) / (||A|| × ||B||)

        Mide el ángulo entre dos vectores en el espacio TF-IDF.
        - 1.0 = documentos idénticos
        - 0.0 = documentos completamente diferentes

    Parámetros:
        title (str): Título de la película de referencia
        df (pd.DataFrame): DataFrame de películas
        cosine_sim (np.ndarray): Matriz de similitud coseno
        n (int): Número de recomendaciones

    Retorna:
        pd.DataFrame: Top-N películas similares con scores
    """
    # Buscar el título (con tolerancia a typos)
    matches = fuzzy_match_title(title, df["title"])
    if not matches:
        print(f"   ⚠️  No se encontró '{title}'. Intenta con otro título.")
        return pd.DataFrame()

    matched_title = matches[0]
    if matched_title.lower() != title.lower():
        print(f"   🔍 ¿Quisiste decir '{matched_title}'?")

    # Obtener el índice de la película
    idx = df[df["title"] == matched_title].index[0]

    # Obtener scores de similitud para esta película
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Ordenar por similitud descendente (excluyendo la película misma)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n + 1]  # Excluir la primera (es ella misma, score=1.0)

    # Obtener los índices de las películas similares
    movie_indices = [i[0] for i in sim_scores]
    similarity_values = [i[1] for i in sim_scores]

    # Construir DataFrame de resultados
    results = df.iloc[movie_indices][["title", "year", "genres_str", "overview", "vote_average"]].copy()
    results["similarity"] = similarity_values
    results = results.reset_index(drop=True)

    return results

=============================================================================
PARTE 2: Metadata Soup + CountVectorizer
=============================================================================

In [ ]:
def build_soup_model(df):
    """
    Construye un modelo de recomendación usando "Metadata Soup":
    una concatenación de géneros + keywords + overview.

    Usa CountVectorizer en vez de TfidfVectorizer porque los metadatos
    (géneros, keywords) no necesitan ponderación IDF — cada mención
    es igualmente importante.

    Parámetros:
        df (pd.DataFrame): DataFrame con columnas necesarias

    Retorna:
        tuple: (count_matrix, cosine_sim_soup, count_vectorizer)
    """
    print("\n🍲 Construyendo modelo con Metadata Soup...")

    # Crear la sopa de metadatos
    df = df.copy()
    df["soup"] = df.apply(create_soup, axis=1)

    print(f"   📝 Ejemplo de soup:")
    sample = df.iloc[0]
    print(f"      Título: {sample['title']}")
    print(f"      Soup: {sample['soup'][:150]}...")

    # Vectorizar con CountVectorizer
    count_vec = CountVectorizer(
        max_features=20000,
        stop_words="english",
        min_df=2,
    )
    count_matrix = count_vec.fit_transform(df["soup"])

    print(f"   📐 Forma de la matriz Count: {count_matrix.shape}")
    print(f"   📝 Vocabulario: {len(count_vec.vocabulary_):,} términos")

    # Similitud coseno
    print("   ⏳ Calculando similitud coseno sobre metadata soup...")
    cosine_sim_soup = cosine_similarity(count_matrix, count_matrix)

    print(f"   ✅ Matriz de similitud: {cosine_sim_soup.shape}")

    return count_matrix, cosine_sim_soup, count_vec, df

=============================================================================
PARTE 3: Comparación TF-IDF vs Metadata Soup
=============================================================================

In [ ]:
def compare_methods(title, df, cosine_sim_tfidf, cosine_sim_soup, n=10):
    """
    Compara las recomendaciones de TF-IDF (solo overview) vs
    Metadata Soup (overview + genres + keywords).

    Parámetros:
        title (str): Título de referencia
        df (pd.DataFrame): DataFrame de películas
        cosine_sim_tfidf (np.ndarray): Similitud basada en TF-IDF
        cosine_sim_soup (np.ndarray): Similitud basada en Metadata Soup
        n (int): Número de recomendaciones por método
    """
    print(f"\n{'=' * 70}")
    print(f"🔬 COMPARACIÓN: TF-IDF vs Metadata Soup para '{title}'")
    print(f"{'=' * 70}")

    matches = fuzzy_match_title(title, df["title"])
    if not matches:
        print(f"   ⚠️  No se encontró '{title}'")
        return

    matched_title = matches[0]
    idx = df[df["title"] == matched_title].index[0]

    # Recomendaciones TF-IDF
    sim_tfidf = sorted(enumerate(cosine_sim_tfidf[idx]), key=lambda x: x[1], reverse=True)[1:n + 1]
    # Recomendaciones Soup
    sim_soup = sorted(enumerate(cosine_sim_soup[idx]), key=lambda x: x[1], reverse=True)[1:n + 1]

    print(f"\n   {'TF-IDF (Solo Overview)':<42} │ {'Metadata Soup (Overview+Genres+KW)'}")
    print(f"   {'─' * 42}┼{'─' * 42}")

    for i in range(n):
        idx_t, score_t = sim_tfidf[i]
        idx_s, score_s = sim_soup[i]

        title_t = df.iloc[idx_t]["title"][:35]
        title_s = df.iloc[idx_s]["title"][:35]

        print(f"   {i + 1:2d}. {title_t:<35} {score_t:.3f} │ {i + 1:2d}. {title_s:<35} {score_s:.3f}")

=============================================================================
PARTE 4: Recomendador con filtros avanzados
=============================================================================

In [ ]:
def get_recommendations_advanced(title, df, cosine_sim, n=10,
                                  genre_filter=None, min_year=None,
                                  min_rating=0.0):
    """
    Recomendador avanzado con soporte para filtros.

    Parámetros:
        title (str): Título de referencia
        df (pd.DataFrame): DataFrame de películas
        cosine_sim (np.ndarray): Matriz de similitud
        n (int): Número de recomendaciones
        genre_filter (str): Filtrar por género (ej: 'Action')
        min_year (int): Año mínimo de lanzamiento
        min_rating (float): Rating mínimo

    Retorna:
        pd.DataFrame: Recomendaciones filtradas
    """
    matches = fuzzy_match_title(title, df["title"])
    if not matches:
        print(f"   ⚠️  No se encontró '{title}'")
        return pd.DataFrame()

    matched_title = matches[0]
    idx = df[df["title"] == matched_title].index[0]

    # Obtener todos los scores de similitud
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:]  # Excluir la película misma

    # Filtrar
    results = []
    for movie_idx, score in sim_scores:
        if len(results) >= n:
            break

        row = df.iloc[movie_idx]

        # Filtro de género
        if genre_filter and genre_filter not in row["genres_list"]:
            continue

        # Filtro de año
        if min_year and (pd.isna(row["year"]) or row["year"] < min_year):
            continue

        # Filtro de rating
        if row["vote_average"] < min_rating:
            continue

        results.append({
            "title": row["title"],
            "year": row["year"],
            "genres": row["genres_str"],
            "rating": row["vote_average"],
            "similarity": score,
        })

    return pd.DataFrame(results)

=============================================================================
PARTE 5: Visualización de similitudes
=============================================================================

In [ ]:
def visualize_similarity_heatmap(titles, df, cosine_sim, method_name="TF-IDF"):
    """
    Genera un heatmap de similitud coseno entre un conjunto de películas.

    Parámetros:
        titles (list[str]): Lista de títulos a comparar
        df (pd.DataFrame): DataFrame de películas
        cosine_sim (np.ndarray): Matriz de similitud
        method_name (str): Nombre del método para el título del gráfico
    """
    ensure_output_dir()

    # Encontrar índices
    indices = []
    valid_titles = []
    for t in titles:
        matches = fuzzy_match_title(t, df["title"])
        if matches:
            idx = df[df["title"] == matches[0]].index[0]
            indices.append(idx)
            valid_titles.append(matches[0])

    if len(indices) < 2:
        print("   ⚠️  Se necesitan al menos 2 películas válidas para el heatmap")
        return

    # Extraer sub-matriz de similitud
    sim_sub = cosine_sim[np.ix_(indices, indices)]

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))

    im = ax.imshow(sim_sub, cmap="magma", vmin=0, vmax=1)

    # Labels
    short_titles = [t[:25] for t in valid_titles]
    ax.set_xticks(range(len(short_titles)))
    ax.set_yticks(range(len(short_titles)))
    ax.set_xticklabels(short_titles, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(short_titles, fontsize=9)

    # Añadir valores numéricos
    for i in range(len(indices)):
        for j in range(len(indices)):
            color = "white" if sim_sub[i, j] < 0.5 else "black"
            ax.text(j, i, f"{sim_sub[i, j]:.2f}", ha="center", va="center",
                    color=color, fontsize=9)

    ax.set_title(f"Similitud Coseno entre Películas ({method_name})")
    plt.colorbar(im, label="Similitud Coseno")

    plt.tight_layout()
    path = f"{OUTPUT_DIR}/02_heatmap_similitud_{method_name.lower().replace(' ', '_')}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   💾 Guardado: {path}")

=============================================================================
Ejecución principal
=============================================================================

In [ ]:
if __name__ == "__main__":
    # Explicación teórica
    explain_tfidf()

    # Cargar datos
    df = load_and_clean_data(include_keywords=True)

    # ─── Método 1: TF-IDF sobre overviews ───
    print("\n" + "=" * 70)
    print("📝 MÉTODO 1: TF-IDF sobre Sinopsis (Overview)")
    print("=" * 70)

    tfidf_matrix, cosine_sim_tfidf, tfidf_vec = build_tfidf_model(df)

    # Probar recomendaciones TF-IDF
    test_movies = ["Toy Story", "The Dark Knight", "The Godfather"]
    for movie in test_movies:
        print(f"\n   🎬 Recomendaciones para '{movie}' (TF-IDF):")
        print("   " + "─" * 60)
        recs = get_recommendations_tfidf(movie, df, cosine_sim_tfidf, n=8)
        if not recs.empty:
            for i, (_, row) in enumerate(recs.iterrows(), 1):
                print(f"   {i:2d}. {row['title']:<35} "
                      f"Sim: {row['similarity']:.3f} │ "
                      f"Rating: {row['vote_average']:.1f}")

    # ─── Método 2: Metadata Soup ───
    print("\n" + "=" * 70)
    print("🍲 MÉTODO 2: Metadata Soup (Genres + Keywords + Overview)")
    print("=" * 70)

    count_matrix, cosine_sim_soup, count_vec, df = build_soup_model(df)

    # Probar recomendaciones Soup
    for movie in test_movies:
        print(f"\n   🎬 Recomendaciones para '{movie}' (Metadata Soup):")
        print("   " + "─" * 60)
        recs = get_recommendations_tfidf(movie, df, cosine_sim_soup, n=8)
        if not recs.empty:
            for i, (_, row) in enumerate(recs.iterrows(), 1):
                print(f"   {i:2d}. {row['title']:<35} "
                      f"Sim: {row['similarity']:.3f} │ "
                      f"Rating: {row['vote_average']:.1f}")

    # ─── Comparación lado a lado ───
    for movie in ["Toy Story", "The Dark Knight"]:
        compare_methods(movie, df, cosine_sim_tfidf, cosine_sim_soup, n=8)

    # ─── Recomendaciones con filtros ───
    print(f"\n{'=' * 70}")
    print("🔧 RECOMENDACIONES CON FILTROS AVANZADOS")
    print(f"{'=' * 70}")

    print("\n   🎬 Películas similares a 'The Dark Knight' (solo Action, post-2000, rating≥7):")
    print("   " + "─" * 60)
    filtered = get_recommendations_advanced(
        "The Dark Knight", df, cosine_sim_soup,
        n=10, genre_filter="Action", min_year=2000, min_rating=7.0
    )
    if not filtered.empty:
        for i, (_, row) in enumerate(filtered.iterrows(), 1):
            year_str = f" ({int(row['year'])})" if pd.notna(row["year"]) else ""
            print(f"   {i:2d}. {row['title']}{year_str:<35} "
                  f"Sim: {row['similarity']:.3f} │ "
                  f"Rating: {row['rating']:.1f}")

    # ─── Heatmap de similitud ───
    print(f"\n{'=' * 70}")
    print("🗺️  HEATMAP DE SIMILITUD")
    print(f"{'=' * 70}")

    comparison_movies = [
        "Toy Story", "Finding Nemo", "Shrek", "The Lion King",  # Animación
        "The Dark Knight", "Batman Begins", "Iron Man",          # Superhéroes
        "The Godfather", "Goodfellas", "Scarface",               # Crimen
        "Titanic", "The Notebook",                               # Romance
    ]

    visualize_similarity_heatmap(comparison_movies, df, cosine_sim_soup, "Metadata Soup")
    visualize_similarity_heatmap(comparison_movies, df, cosine_sim_tfidf, "TF-IDF")

    print(f"\n{'=' * 70}")
    print("✅ Módulo 02 completado. Revisa las gráficas en output/")
    print(f"{'=' * 70}")